In [7]:
import pandas as pd
import numpy as np
df = pd.read_csv("METABRIC_RNA_Mutation.csv", low_memory=False)
print(df.shape)
df.head()

(1904, 693)


,patient_id,age_at_diagnosis,type_of_breast_surgery,cancer_type,cancer_type_detailed,cellularity,chemotherapy,pam50_+_claudin-low_subtype,cohort,er_status_measured_by_ihc,...,mtap_mut,ppp2cb_mut,smarcd1_mut,nras_mut,ndfip1_mut,hras_mut,prps2_mut,smarcb1_mut,stmn2_mut,siah1_mut
0,0,75.65,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,NaN,0,claudin-low,1.0,Positve,...,0,0,0,0,0,0,0,0,0,0
1,2,43.19,BREAST CONSERVING,Breast Cancer,Breast Invasive Ductal Carcinoma,High,0,LumA,1.0,Positve,...,0,0,0,0,0,0,0,0,0,0
2,5,48.87,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,1,LumB,1.0,Positve,...,0,0,0,0,0,0,0,0,0,0
3,6,47.68,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Moderate,1,LumB,1.0,Positve,...,0,0,0,0,0,0,0,0,0,0
4,8,76.97,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,High,1,LumB,1.0,Positve,...,0,0,0,0,0,0,0,0,0,0


Select Predictors

In [8]:
cols = [
    "overall_survival_months",
    "age_at_diagnosis",
    "lymph_nodes_examined_positive",
    "mutation_count",
    "tumor_size",
    "neoplasm_histologic_grade",
    "nottingham_prognostic_index",
    "cellularity",
    "er_status",
    "her2_status",
    "chemotherapy",
    "hormone_therapy"
]

data = df[cols].copy()

Missing Values

In [9]:
data = data.dropna()
print(data.shape)

(1727, 12)


Encode Categorical Variables

In [10]:
data_encoded = pd.get_dummies(data, drop_first=True)
print(data_encoded.shape)
data_encoded.head()

(1727, 13)


,overall_survival_months,age_at_diagnosis,lymph_nodes_examined_positive,mutation_count,tumor_size,neoplasm_histologic_grade,nottingham_prognostic_index,chemotherapy,hormone_therapy,cellularity_Low,cellularity_Moderate,er_status_Positive,her2_status_Positive
1,84.633333,43.19,0.0,2.0,10.0,3.0,4.020,0,1,False,False,True,False
2,163.700000,48.87,1.0,2.0,15.0,2.0,4.030,1,1,False,False,True,False
3,164.933333,47.68,3.0,1.0,25.0,2.0,4.050,1,1,False,True,True,False
4,41.366667,76.97,8.0,2.0,40.0,3.0,6.080,1,1,False,False,True,False
5,7.800000,78.77,0.0,4.0,31.0,3.0,4.062,0,1,False,True,True,False


Separate predictors and response

In [15]:
y = pd.to_numeric(data_encoded["overall_survival_months"], errors="coerce")
X = data_encoded.drop(columns=["overall_survival_months"])

X = X.apply(pd.to_numeric, errors="coerce")
X = X.astype(float)
y = y.astype(float)

In [16]:
combined = pd.concat([y, X], axis=1).dropna()

y = combined["overall_survival_months"]
X = combined.drop(columns=["overall_survival_months"])

In [17]:
X_full = sm.add_constant(X)
full_model = sm.OLS(y, X_full).fit()

In [18]:
cols = [
    "overall_survival_months",
    "age_at_diagnosis",
    "lymph_nodes_examined_positive",
    "mutation_count",
    "tumor_size",
    "neoplasm_histologic_grade",
    "nottingham_prognostic_index",
    "cellularity",
    "er_status",
    "her2_status",
    "chemotherapy",
    "hormone_therapy"
]

data = df[cols].copy()
data = data.dropna()

data_encoded = pd.get_dummies(data, drop_first=True)

y = pd.to_numeric(data_encoded["overall_survival_months"], errors="coerce")
X = data_encoded.drop(columns=["overall_survival_months"])

X = X.apply(pd.to_numeric, errors="coerce").astype(float)
y = y.astype(float)

combined = pd.concat([y, X], axis=1).dropna()

y = combined["overall_survival_months"]
X = combined.drop(columns=["overall_survival_months"])

print(X.dtypes)
print(y.dtypes)
print(X.shape, y.shape)

age_at_diagnosis                 float64
lymph_nodes_examined_positive    float64
mutation_count                   float64
tumor_size                       float64
neoplasm_histologic_grade        float64
nottingham_prognostic_index      float64
chemotherapy                     float64
hormone_therapy                  float64
cellularity_Low                  float64
cellularity_Moderate             float64
er_status_Positive               float64
her2_status_Positive             float64
dtype: object
float64
(1727, 12) (1727,)


In [19]:
X_full = sm.add_constant(X)
full_model = sm.OLS(y, X_full).fit()
print(full_model.summary())

                               OLS Regression Results                              
Dep. Variable:     overall_survival_months   R-squared:                       0.147
Model:                                 OLS   Adj. R-squared:                  0.141
Method:                      Least Squares   F-statistic:                     24.62
Date:                     Sat, 07 Mar 2026   Prob (F-statistic):           1.84e-51
Time:                             19:34:37   Log-Likelihood:                -9801.6
No. Observations:                     1727   AIC:                         1.963e+04
Df Residuals:                         1714   BIC:                         1.970e+04
Df Model:                               12                                         
Covariance Type:                 nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------